# Longitudinal API Evolution and Breaking Changes

This notebook reproduces the longitudinal analyses reported in Section 5.3:

- the corpus-wide library-profile ridge figure;
- the public versus excluded/internal breaking-change table; and
- one API-evolution timeline per studied library, including Guava's paper figure.


In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="talk")

DATA_DIR = Path("data")
FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

NUMERIC_COLUMNS = [
    "all_api_symbols_count",
    "exported_types_count",
    "exported_methods_count",
    "exported_fields_count",
    "exported_symbols_count",
    "deprecated_count",
    "internal_count",
    "api_breaking_changes_count",
    "excluded_breaking_changes_count",
]

commit_frames = []
for path in sorted(DATA_DIR.glob("*-commits.csv")):
    frame = pd.read_csv(path, low_memory=False)
    frame["library"] = frame["library"].fillna(path.stem.removesuffix("-commits"))
    commit_frames.append(frame)

commits_df = pd.concat(commit_frames, ignore_index=True)
commits_df["date_utc"] = pd.to_datetime(commits_df["date_utc"], errors="coerce", utc=True)
for column in NUMERIC_COLUMNS:
    if column not in commits_df.columns:
        commits_df[column] = 0
    commits_df[column] = pd.to_numeric(commits_df[column], errors="coerce").fillna(0)

exported_symbol_fallback = (
    commits_df["exported_types_count"]
    + commits_df["exported_methods_count"]
    + commits_df["exported_fields_count"]
)
commits_df["exported_symbols_count"] = commits_df["exported_symbols_count"].where(
    commits_df["exported_symbols_count"] > 0, exported_symbol_fallback
)
commits_df = commits_df.sort_values(["library", "date_utc", "commit_sha"]).reset_index(drop=True)

print(f"Libraries: {commits_df['library'].nunique()}")
print(f"Commit rows: {len(commits_df):,}")


## Corpus-wide profiles and breakage scope

The ridge plot normalizes each history from its first to last commit. Blue ridges show exported API size relative to that library's peak; red bars show public-API breaking-change intensity in 30 equal history bins. The table reproduces the selection described in the paper: the five largest excluded/internal shares among histories with at least 500 detected BCs, followed by the three largest public-break histories with no excluded/internal BCs.


In [ ]:
profile_rows = []
timeline_frames = []

for library, history in commits_df.groupby("library", sort=False):
    history = history.sort_values(["date_utc", "commit_sha"]).reset_index(drop=True).copy()
    history["history_pct"] = 0.0 if len(history) == 1 else history.index / (len(history) - 1)
    peak_exported = float(history["exported_symbols_count"].max())
    history["api_relative_to_peak"] = (
        history["exported_symbols_count"] / peak_exported if peak_exported else 0.0
    )
    timeline_frames.append(history)

    public_breaks = float(history["api_breaking_changes_count"].sum())
    excluded_breaks = float(history["excluded_breaking_changes_count"].sum())
    total_breaks = public_breaks + excluded_breaks
    break_center = (
        float((history["history_pct"] * history["api_breaking_changes_count"]).sum() / public_breaks)
        if public_breaks
        else np.nan
    )
    profile_rows.append(
        {
            "library": library,
            "peak_exported": peak_exported,
            "public_breaks": public_breaks,
            "excluded_breaks": excluded_breaks,
            "total_breaks": total_breaks,
            "excluded_share": excluded_breaks / total_breaks if total_breaks else 0.0,
            "break_center": break_center,
        }
    )

timelines_df = pd.concat(timeline_frames, ignore_index=True)
profile_df = pd.DataFrame(profile_rows).set_index("library")

scope_table = pd.concat(
    [
        profile_df[profile_df["total_breaks"] >= 500]
        .nlargest(5, "excluded_share"),
        profile_df[profile_df["excluded_breaks"] == 0].nlargest(3, "public_breaks"),
    ]
)[["public_breaks", "excluded_breaks", "total_breaks", "excluded_share"]]
display(scope_table.round({"public_breaks": 0, "excluded_breaks": 0, "total_breaks": 0, "excluded_share": 3}))

BIN_COUNT = 30
library_order = profile_df.sort_values(["break_center", "public_breaks"], ascending=[True, False]).index.tolist()
timelines_df["history_bin"] = np.clip(
    np.floor(timelines_df["history_pct"] * BIN_COUNT).astype(int), 0, BIN_COUNT - 1
)
api_profile = (
    timelines_df.groupby(["library", "history_bin"])["api_relative_to_peak"]
    .mean()
    .unstack(fill_value=0)
    .reindex(index=library_order, columns=range(BIN_COUNT), fill_value=0)
)
break_intensity = (
    timelines_df.groupby(["library", "history_bin"])["api_breaking_changes_count"]
    .sum()
    .unstack(fill_value=0)
    .reindex(index=library_order, columns=range(BIN_COUNT), fill_value=0)
)
break_intensity = break_intensity.div(
    break_intensity.max(axis=1).replace(0, np.nan), axis=0
).fillna(0)


In [ ]:
ridge_kernel = np.array([1, 2, 3, 2, 1], dtype=float)
ridge_kernel /= ridge_kernel.sum()
ridge_x = np.linspace(0, 1, BIN_COUNT)
ridge_edges = np.linspace(0, 1, BIN_COUNT + 1)
bar_centers = (ridge_edges[:-1] + ridge_edges[1:]) / 2
bar_width = 0.86 / BIN_COUNT
ridge_api_profile = pd.DataFrame(
    np.vstack([np.convolve(api_profile.loc[library], ridge_kernel, mode="same") for library in library_order]),
    index=library_order,
)

split_mid = int(np.ceil(len(library_order) / 2))
library_groups = [library_order[:split_mid], library_order[split_mid:]]
ridge_spacing = 1.15
api_scale = 0.58
break_scale = 0.44
split_max_rows = max(len(group) for group in library_groups)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10.6, max(7.5, 0.65 * split_max_rows)),
    sharex=True,
    gridspec_kw={"wspace": 0.03},
)
for panel_index, (ax, libraries) in enumerate(zip(axes, library_groups)):
    local_y = (np.arange(len(libraries))[::-1] * ridge_spacing).astype(float)
    label_x, alignment = [(0.05, "left"), (0.95, "right")][panel_index]
    label_index = np.abs(ridge_x - label_x).argmin()
    for band_index, baseline in enumerate(local_y):
        if band_index % 2 == 0:
            ax.axhspan(baseline - 0.35, baseline + 0.78, color="#f4f6f8", alpha=0.55, zorder=0)
    for guide in [0.25, 0.5, 0.75]:
        ax.axvline(guide, color="#d9d9d9", linewidth=0.9, linestyle=(0, (2, 3)), zorder=1)
    for row, library in enumerate(libraries):
        baseline = local_y[row]
        api_height = api_scale * ridge_api_profile.loc[library].to_numpy()
        break_bins = break_intensity.loc[library].to_numpy()
        nonzero = break_bins > 0
        ax.hlines(baseline, 0, 1, color="#7f8c8d", linewidth=0.35, alpha=0.45, zorder=2)
        ax.fill_between(ridge_x, baseline + 0.02, baseline + 0.02 + api_height, color="#7ea0c4", alpha=0.66, linewidth=0, zorder=3)
        ax.plot(ridge_x, baseline + 0.02 + api_height, color="#2f5d7c", linewidth=1.05, zorder=4)
        ax.text(label_x, baseline + 0.02 + max(0.06, 0.55 * api_height[label_index]), library, ha=alignment, va="center", fontsize=7.3, color="#203040", fontweight="semibold", bbox={"boxstyle": "round,pad=0.18", "facecolor": "white", "edgecolor": "none", "alpha": 0.82}, zorder=6)
        if nonzero.any():
            heights = break_scale * break_bins[nonzero]
            ax.bar(bar_centers[nonzero], heights, width=bar_width, bottom=baseline - heights - 0.03, color="#d36b5d", edgecolor="#8b1e1e", linewidth=0.35, alpha=0.94, zorder=5)
    ax.set_yticks([])
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.7, (split_max_rows - 1) * ridge_spacing + 1.05)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["First commit", "Last commit"], fontsize=8)
    ax.get_xticklabels()[0].set_ha("left")
    ax.get_xticklabels()[1].set_ha("right")
    ax.grid(axis="x", alpha=0.12)
    ax.tick_params(axis="y", left=False, right=False, labelleft=False, labelright=False)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)

fig.savefig(FIGURE_DIR / "libs-ridge.pdf", bbox_inches="tight", pad_inches=0.02)
plt.show()
plt.close(fig)


## Per-library timelines

Each timeline shows exported API symbols, excluded API symbols, and public and excluded/internal breaking changes. The Guava output is the timeline included in the paper; the remaining 28 are retained as replication-package data products.


In [ ]:
import matplotlib.dates as mdates

API_LINE_COLOR = "#0072B2"
DEPRECATED_LINE_COLOR = "#CC79A7"
INTERNAL_LINE_COLOR = "#595959"
PUBLIC_BREAK_COLOR = "#D55E00"
EXCLUDED_BREAK_COLOR = "#E69F00"

FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

library_order = sorted(commits_df["library"].dropna().unique())

for lib in library_order:
    lib_df = commits_df.loc[commits_df["library"] == lib].copy()
    lib_df = lib_df.sort_values(["date_utc", "commit_sha"]).reset_index(drop=True)
    has_api = lib_df["all_api_symbols_count"] > 0
    start_idx = int(has_api.idxmax()) if has_api.any() else 0
    plot_df = lib_df.loc[start_idx:].copy()

    if plot_df.empty:
        continue

    release_mask = plot_df["tag"].fillna("").astype(str).str.match(
        r"^v?\d+(?:\.\d+){0,2}$"
    )
    release_df = plot_df.loc[release_mask, ["date_utc", "tag"]].drop_duplicates("tag")

    fig, ax1 = plt.subplots(figsize=(16, 7))

    ax1.plot(
        plot_df["date_utc"],
        plot_df["exported_symbols_count"],
        color=API_LINE_COLOR,
        linewidth=2.3,
        label="Exported API symbols",
    )
    ax1.plot(
        plot_df["date_utc"],
        plot_df["deprecated_count"],
        color=DEPRECATED_LINE_COLOR,
        linewidth=1.9,
        linestyle="--",
        label="Excluded API (deprecated)",
    )
    ax1.plot(
        plot_df["date_utc"],
        plot_df["internal_count"],
        color=INTERNAL_LINE_COLOR,
        linewidth=1.9,
        linestyle=":",
        label="Excluded API (internal)",
    )
    ax1.fill_between(
        plot_df["date_utc"],
        0,
        plot_df["exported_symbols_count"],
        color=API_LINE_COLOR,
        alpha=0.12,
    )
    ax1.set_xlabel("Date")
    ax1.set_ylabel("API symbols")
    ax1.set_title(f"{lib}: exported API, excluded API, and breaking changes")
    ax1.xaxis.set_major_locator(mdates.YearLocator(base=2))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax1.grid(axis="y", alpha=0.25)
    ax1.set_xlim(plot_df["date_utc"].min(), plot_df["date_utc"].max())
    ax1.set_ylim(bottom=0)
    ax1.margins(x=0, y=0)

    ax2 = ax1.twinx()
    public_breaks = plot_df.loc[plot_df["api_breaking_changes_count"] > 0]
    excluded_breaks = plot_df.loc[plot_df["excluded_breaking_changes_count"] > 0]
    ax2.vlines(
        public_breaks["date_utc"],
        0,
        public_breaks["api_breaking_changes_count"],
        color=PUBLIC_BREAK_COLOR,
        alpha=0.9,
        linewidth=1.2,
        label="Public-API breaking changes",
    )
    ax2.vlines(
        excluded_breaks["date_utc"],
        0,
        excluded_breaks["excluded_breaking_changes_count"],
        color=EXCLUDED_BREAK_COLOR,
        alpha=0.9,
        linewidth=1.2,
        label="Excluded/internal breaking changes",
    )
    ax2.set_ylabel("Breaking changes per commit")
    ax2.set_ylim(bottom=0)
    ax2.margins(x=0, y=0)
    ax2.grid(False)

    if not release_df.empty:
        for row in release_df.itertuples(index=False):
            ax1.axvline(row.date_utc, color="#c0c0c0", linestyle=":", linewidth=1.0, zorder=0)
            ax1.text(
                row.date_utc,
                0.99,
                row.tag,
                rotation=90,
                ha="right",
                va="top",
                fontsize=9,
                color="#6b7280",
                transform=ax1.get_xaxis_transform(),
            )

    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(
        handles1 + handles2,
        labels1 + labels2,
        loc="upper left",
        frameon=True,
        fontsize=10,
    )

    plt.tight_layout()
    figure_path = FIGURE_DIR / f"{lib}_api_excluded_breaking_changes.pdf"
    fig.savefig(figure_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)
